# GA-STOMP on CWRU bearing data

Builds the CUDA binary and runs one experiment (`EXPERIMENT` in the settings cell) on
CWRU 12 kHz drive-end signals,
once per fitness variant (set `EXPERIMENT` below), and writes everything to
`/kaggle/working/results/<variant>/<dataset>/` plus `results/figures/ablation.csv`
and `RUN_STATUS.txt`.

Run it from the Kaggle UI, or let the GitHub Actions workflow push it with `kaggle kernels push`
(the workflow replaces `GIT_REF` below with the commit SHA it is testing).

Expected window = 12000 / fault frequency, using each file's recorded RPM:
BPFI = 5.4152 x shaft Hz, BPFO = 3.5848 x shaft Hz.
At 1797 rpm (29.95 Hz): inner race 74 samples, outer race 112 samples.

In [ ]:
# ---- Settings ---------------------------------------------------------------
GIT_REF  = "v6-spacing-regularity"   # branch, tag or SHA; CI overwrites this line
REPO_URL = "https://github.com/HoaPNG520/Motif-finding-GA-STOMP.git"
CWRU_SLUG = "sufian79/cwru-mat-full-dataset"
FS        = 12000       # Hz, CWRU 12k drive-end data
SEG_LEN   = 6000        # samples per segment

# CWRU 12k drive-end files: number -> (label, class, fault size in mils, load HP)
# Outer-race files are the 6 o'clock (centred) position.
CWRU_FILES = {}
for load, num in enumerate((97, 98, 99, 100)):
    CWRU_FILES[num] = ("Normal", "Normal", 0, load)
for cls, size, first in (("IR", 7, 105), ("IR", 14, 169), ("IR", 21, 209),
                         ("B",  7, 118), ("B",  14, 185), ("B",  21, 222),
                         ("OR", 7, 130), ("OR", 14, 197), ("OR", 21, 234)):
    for load in range(4):
        CWRU_FILES[first + load] = (f"{cls}{size:03d}", cls, size, load)

NOMINAL_RPM  = {0: 1797, 1: 1772, 2: 1750, 3: 1730}   # used only if a file has no RPM
FAULT_ORDERS = {"IR": 5.4152, "OR": 3.5848}            # BPFI / BPFO per shaft revolution
# Ball faults get no expected window: whether the impact period is 1/BSF or
# 1/(2 BSF) is ambiguous on CWRU, so they are excluded from window accuracy.

# Experiments.  files: CWRU numbers; segments: 6000-sample slices per file;
# variants: extra config lines per variant (results/<variant>/<name>/).
NO_FFT = "w_period=0\nperiod_bias_init=0"
EXPERIMENTS = {
    "ablation": dict(
        files=[105, 106, 130, 100], segments=1, keep_bulky=True, per_run_figures=True,
        variants={
            "full":          "",
            "no_period":     NO_FFT,
            "no_regularity": "w_regularity=0",
            "no_contrast":   "w_contrast=0",
        }),
    "null_contrast": dict(
        files=[105, 106, 130, 100], segments=1, keep_bulky=True, per_run_figures=True,
        variants={
            "full":         "",
            "contrast_raw": NO_FFT + "\nw_contrast=1\nw_regularity=0",
            "null_shuffle": NO_FFT + "\nw_contrast=1\nw_regularity=0\nnull_correct=1\nnull_surrogate=shuffle",
            "null_phase":   NO_FFT + "\nw_contrast=1\nw_regularity=0\nnull_correct=1\nnull_surrogate=phase",
        }),
    # Healthy vs faulty on every 12k drive-end file: full pipeline with the
    # phase-corrected contrast logged; one GA run per segment (no seed repeats).
    "fault_detection": dict(
        files=sorted(CWRU_FILES), segments=3, keep_bulky=False, per_run_figures=False,
        variants={"detect": "null_correct=1\nnull_surrogate=phase\nn_seeds=0"}),
}
EXPERIMENT = "fault_detection"
EXP = EXPERIMENTS[EXPERIMENT]
VARIANTS = EXP["variants"]

CONFIG_TEMPLATE = """
m_min=20
m_max=200
ez_min=0.25
ez_max=1.0
k_min=2
k_max=20
population=20
generations=20
tournament_k=3
mutation_rate=0.30
elite_count=2
approx_frac=0.5
n_seeds=3
verbose=1
sampling_rate={fs}
fault_freq_min_hz=50
fault_freq_max_hz=300
landscape_step=5
input_path={input_path}
output_dir={output_dir}
{extra}
"""

import os, subprocess, shutil, glob
SRC_DIR  = "/tmp/GA_STOMP"           # repo checkout (kept out of /kaggle/working)
DATA_DIR = "/tmp/ga_data"
OUT_DIR  = "/kaggle/working/results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
status = {}                          # step -> "OK" / "FAIL: reason"

def sh(cmd, **kw):
    """Run a command, echo it, never raise (so results are always saved)."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), text=True,
                       capture_output=True, **kw)
    if r.stdout: print(r.stdout[-4000:])
    if r.stderr: print("STDERR:", r.stderr[-4000:])
    return r

In [ ]:
# ---- Kaggle API credentials (optional) --------------------------------------
# Only needed if the CWRU dataset is not attached to this notebook and has to be
# downloaded. Add a Kaggle secret named KAGGLE_API_TOKEN (and optionally
# KAGGLE_USERNAME) under Add-ons -> Secrets. Nothing is written to disk.
have_api = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME"):
        try:
            val = secrets.get_secret(name)
            if val:
                os.environ[name] = val
                print(f"[secrets] {name} loaded")
        except Exception:
            print(f"[secrets] {name} not attached -- skipped")
    have_api = bool(os.environ.get("KAGGLE_API_TOKEN"))
except ImportError:
    print("[secrets] not running on Kaggle -- using environment as-is")
    have_api = bool(os.environ.get("KAGGLE_API_TOKEN"))
print("Kaggle API available:", have_api)

In [ ]:
# ---- Clone the repo at GIT_REF ----------------------------------------------
if os.path.exists(SRC_DIR):
    shutil.rmtree(SRC_DIR)
r1 = sh(["git", "clone", "--quiet", REPO_URL, SRC_DIR])
r2 = sh(["git", "-C", SRC_DIR, "checkout", "--quiet", GIT_REF]) if r1.returncode == 0 else r1
status["clone"] = "OK" if r2.returncode == 0 else f"FAIL: git ({GIT_REF})"
if status["clone"] == "OK":
    sh(["git", "-C", SRC_DIR, "log", "--oneline", "-3"])
print("clone:", status["clone"])

In [ ]:
# ---- Detect GPU architecture and build --------------------------------------
gpu = sh("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader")
sm_arch = "sm_75"                    # T4 default (P100 is retired on Kaggle)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, cap = [s.strip() for s in gpu.stdout.strip().splitlines()[0].split(",")]
    sm_arch = "sm_" + cap.replace(".", "")
    print(f"GPU: {name}  ->  {sm_arch}")
else:
    print("No GPU visible -- enable a GPU accelerator for this notebook")

if status["clone"] == "OK":
    b = sh(f"make -C {SRC_DIR} SM_ARCH={sm_arch} -j4")
    ok = b.returncode == 0 and os.path.exists(f"{SRC_DIR}/ga_stomp")
    status["build"] = "OK" if ok else "FAIL: nvcc"
    with open(f"{OUT_DIR}/build.log", "w") as f:
        f.write(b.stdout + "\n" + b.stderr)
else:
    status["build"] = "SKIPPED: no checkout"
print("build:", status["build"])

In [ ]:
# ---- Locate (or download) CWRU, cut segments, record RPM and baselines ------
import numpy as np, scipy.io, scipy.stats, csv

def find_mat(fname):
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True) + \
           glob.glob(f"{DATA_DIR}/cwru/**/{fname}", recursive=True)
    return hits[0] if hits else None

if find_mat("105.mat") is None and have_api:
    print("CWRU not attached -- downloading via Kaggle API")
    sh(["kaggle", "datasets", "download", "-d", CWRU_SLUG,
        "-p", f"{DATA_DIR}/cwru", "--unzip", "-q"])

META = {}          # run name -> metadata row
for num in EXP["files"]:
    label, cls, size, load = CWRU_FILES[num]
    path = find_mat(f"{num}.mat")
    if path is None:
        status[f"data:{num}"] = f"FAIL: {num}.mat not found (attach {CWRU_SLUG} or add KAGGLE_API_TOKEN)"
        continue
    m = scipy.io.loadmat(path)
    # Some files hold more than one recording (99.mat also has X098_*), so
    # prefer the key that matches the file number.
    de = f"X{num:03d}_DE_time"
    if de not in m:
        de = next((k for k in m if k.endswith("DE_time")), None)
    if de is None:
        status[f"data:{num}"] = f"FAIL: no DE_time key in {num}.mat"
        continue
    rpm_key = f"X{num:03d}RPM"
    if rpm_key in m:
        rpm, rpm_src = float(np.ravel(m[rpm_key])[0]), "file"
    else:
        rpm, rpm_src = float(NOMINAL_RPM[load]), "nominal"
    sig = np.ravel(m[de]).astype(np.float64)
    order = FAULT_ORDERS.get(cls)
    expected = int(round(FS / (order * rpm / 60.0))) if order else None
    status[f"data:{num}"] = "OK"
    for seg in range(EXP["segments"]):
        part = sig[seg * SEG_LEN:(seg + 1) * SEG_LEN]
        if len(part) < SEG_LEN:
            break
        name = f"{label}_{load}hp" + (f"_s{seg}" if EXP["segments"] > 1 else "")
        out = f"{DATA_DIR}/{name}.csv"
        np.savetxt(out, part, fmt="%.8f")
        META[name] = dict(name=name, file=num, label=label, cls=cls, size=size, load=load,
                          segment=seg, rpm=round(rpm, 1), rpm_source=rpm_src,
                          de_key=de, expected_m=expected if expected else "",
                          kurtosis=float(scipy.stats.kurtosis(part)),
                          rms=float(np.sqrt(np.mean(part ** 2))), csv=out)
    print(f"{num:>4}.mat  {label:>7} {load}hp  rpm={rpm:.0f} ({rpm_src})  key={de}  expected m={expected}")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
with open(f"{OUT_DIR}/figures/files.csv", "w", newline="") as f:
    cols = [c for c in next(iter(META.values())) if c != "csv"] if META else []
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    w.writerows(META.values())
print(f"{len(META)} segments from {len(EXP['files'])} files")

In [ ]:
# ---- Run every variant on every segment -------------------------------------
# Layout: results/<variant>/<name>/
import time
t0 = time.time()
for variant, extra in VARIANTS.items():
    for i, (name, meta) in enumerate(sorted(META.items())):
        key = f"run:{variant}/{name}"
        if status.get("build") != "OK":
            status[key] = "SKIPPED"
            continue
        out_dir = f"{OUT_DIR}/{variant}/{name}"
        os.makedirs(out_dir, exist_ok=True)
        cfg_path = f"{out_dir}/input_config.ini"
        with open(cfg_path, "w") as f:
            f.write(CONFIG_TEMPLATE.format(fs=FS, input_path=meta["csv"],
                                           output_dir=out_dir, extra=extra))
        r = subprocess.run([f"{SRC_DIR}/ga_stomp", cfg_path], text=True, capture_output=True)
        with open(f"{out_dir}/stdout.log", "w") as f:
            f.write(r.stdout + ("\nSTDERR:\n" + r.stderr if r.stderr else ""))
        ok = r.returncode == 0 and os.path.exists(f"{out_dir}/summary.csv")
        status[key] = "OK" if ok else f"FAIL: exit {r.returncode}"
        if not EXP["keep_bulky"]:
            for bulky in ("series.csv", "matrix_profile.csv"):
                if os.path.exists(f"{out_dir}/{bulky}"):
                    os.remove(f"{out_dir}/{bulky}")
        best = next((l.strip() for l in r.stdout.splitlines() if "Best window_size" in l), "")
        print(f"[{time.time() - t0:6.0f}s] {variant:>13} {name:>16} "
              f"(expected {meta['expected_m'] or '-'}): {status[key]}  {best}")
        if not ok:
            print(r.stdout[-2000:], r.stderr[-2000:])

In [ ]:
# ---- Verify the GPU matrix profile against a NumPy brute force --------------
# Pins m = 74 and ez_factor = 0.25 (ez = floor(0.25 * 74) = 18, pairs with
# j - i >= ez), runs the binary once, and compares matrix_profile.csv with an
# exact float64 computation on the same segment.
def verify_stomp(m=74, ez_factor=0.25, tol=0.05):
    name = next((n for n in sorted(META) if META[n]["cls"] == "IR"), next(iter(sorted(META))))
    out_dir = f"{OUT_DIR}/verify/{name}"
    os.makedirs(out_dir, exist_ok=True)
    cfg = CONFIG_TEMPLATE.format(fs=FS, input_path=META[name]["csv"], output_dir=out_dir,
                                 extra=f"m_min={m}\nm_max={m}\nez_min={ez_factor}\n"
                                       f"ez_max={ez_factor}\npopulation=4\ngenerations=1\n"
                                       "n_seeds=0\nlandscape_step=0")
    with open(f"{out_dir}/input_config.ini", "w") as f:
        f.write(cfg)
    r = subprocess.run([f"{SRC_DIR}/ga_stomp", f"{out_dir}/input_config.ini"],
                       text=True, capture_output=True)
    if r.returncode != 0 or not os.path.exists(f"{out_dir}/matrix_profile.csv"):
        return f"FAIL: binary exit {r.returncode}"
    with open(f"{out_dir}/matrix_profile.csv") as f:
        mp_rows = list(csv.DictReader(l for l in f if not l.startswith("#")))
    gpu = np.array([float(r["mp_distance"]) for r in mp_rows])
    gpu_idx = np.array([int(r["mp_index"]) for r in mp_rows])

    x = np.loadtxt(META[name]["csv"])
    ez = max(1, int(np.floor(np.float32(ez_factor) * np.float32(m))))
    W = np.lib.stride_tricks.sliding_window_view(x, m)
    Z = (W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)
    L = len(Z)
    ref = np.empty(L)
    ref_idx = np.empty(L, dtype=int)
    for a in range(0, L, 1000):
        b = min(L, a + 1000)
        d2 = 2.0 * m * (1.0 - np.clip(Z[a:b] @ Z.T / m, -1.0, 1.0))
        rows_i = np.arange(a, b)[:, None]
        d2[np.abs(rows_i - np.arange(L)[None, :]) < ez] = np.inf
        ref_idx[a:b] = d2.argmin(1)
        ref[a:b] = np.sqrt(np.maximum(d2.min(1), 0.0))
    if len(gpu) != L:
        return f"FAIL: length {len(gpu)} vs {L}"
    err = np.abs(gpu - ref)
    same_idx = np.mean(gpu_idx == ref_idx)
    msg = (f"max_abs_err={err.max():.2e} median={np.median(err):.2e} "
           f"index_match={100 * same_idx:.1f}% (m={m}, ez={ez}, {name})")
    print("[verify]", msg)
    with open(f"{out_dir}/verify.txt", "w") as f:
        f.write(msg + "\n")
    return ("OK: " if err.max() < tol else "FAIL: ") + msg

if status.get("build") == "OK" and META:
    try:
        status["verify:stomp"] = verify_stomp()
    except Exception as e:
        status["verify:stomp"] = f"FAIL: {type(e).__name__}: {e}"
print("verify:", status.get("verify:stomp"))

In [ ]:
# ---- Figures, tables, status file -------------------------------------------
expected = [f"{n}={m['expected_m']}" for n, m in META.items() if m["expected_m"]]
plot_ok = True

if EXP["per_run_figures"]:
    for variant in VARIANTS:
        run_dirs = [f"{OUT_DIR}/{variant}/{n}" for n in sorted(META)
                    if status.get(f"run:{variant}/{n}") == "OK"]
        if run_dirs:
            p = sh(["python", f"{SRC_DIR}/plot_results.py", *run_dirs, "--expected", *expected,
                    "--out", f"{OUT_DIR}/{variant}/figures"])
            plot_ok &= p.returncode == 0
    p = sh(["python", f"{SRC_DIR}/plot_results.py", "--ablation", OUT_DIR, "--expected", *expected])
    plot_ok &= p.returncode == 0

def build_detection_table():
    # One row per segment: file metadata + GA summary + landscape statistics.
    rows = []
    for variant in VARIANTS:
        for name, meta in sorted(META.items()):
            d = f"{OUT_DIR}/{variant}/{name}"
            if status.get(f"run:{variant}/{name}") != "OK":
                continue
            with open(f"{d}/summary.csv") as f:
                srow = next(csv.DictReader(f))
            with open(f"{d}/landscape.csv") as f:
                lrows = list(csv.DictReader(l for l in f if not l.startswith("#")))
            land = {k: np.array([float(r[k]) for r in lrows]) for k in lrows[0]}
            exp_m = meta["expected_m"]
            best_m = int(srow["best_m"])
            row = {k: v for k, v in meta.items() if k != "csv"}
            row.update(variant=variant, best_m=best_m,
                       detected_period=int(srow["detected_period"]),
                       abs_error=abs(best_m - exp_m) if exp_m else "",
                       rel_error=round(abs(best_m - exp_m) / exp_m, 4) if exp_m else "",
                       fitness=float(srow["fitness"]),
                       contrast_raw=float(srow["contrast_raw"]),
                       contrast_null=float(srow["contrast_null"]),
                       contrast_corr=float(srow["contrast"]),
                       land_raw_mean=float(np.nanmean(land["contrast_raw"])),
                       land_corr_mean=float(np.nanmean(land["contrast"])),
                       land_corr_max=float(np.nanmax(land["contrast"])))
            rows.append(row)
    det = f"{OUT_DIR}/figures/detection.csv"
    if not rows:
        return False
    with open(det, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    p = sh(["python", f"{SRC_DIR}/plot_results.py", "--detection", det,
            "--out", f"{OUT_DIR}/figures"])
    return p.returncode == 0

if EXPERIMENT == "fault_detection":
    try:
        plot_ok &= build_detection_table()
    except Exception as e:          # keep going so RUN_STATUS.txt is written
        import traceback
        traceback.print_exc()
        status["detection_table"] = f"FAIL: {type(e).__name__}: {e}"
        plot_ok = False

status["plots"] = "OK" if plot_ok else "FAIL: plotting / tables"

for table in ("ablation.csv", "detection_auc.csv", "window_accuracy.csv"):
    path = f"{OUT_DIR}/figures/{table}"
    if os.path.exists(path):
        print(f"== {table}")
        print(open(path).read())

with open(f"{OUT_DIR}/RUN_STATUS.txt", "w") as f:
    f.write(f"git_ref={GIT_REF}\nexperiment={EXPERIMENT}\n")
    for k, v in status.items():
        f.write(f"{k}={v}\n")
print(open(f"{OUT_DIR}/RUN_STATUS.txt").read())